In [0]:
from pyspark.sql.functions import to_date, current_date, col

# Step-1: Read from Bronze (Streaming)
# 
cdc_df = spark.readStream.table("retails.silver.customers_cdc")


In [0]:
from pyspark.sql.functions import to_json, struct, when, col

# Step 4: Apply Type Casting & Standardization
cdc_df = cdc_df.withColumn("customer_id", col("customer_id").cast("integer")) \
                    .withColumn("customer_fname", col("customer_fname").cast("string")) \
                    .withColumn("customer_lname", col("customer_lname").cast(("string"))) \
                    .withColumn("customer_email", col("customer_email").cast("string")) \
                    .withColumn("customer_password", col("customer_password").cast("string")) \
                    .withColumn("customer_street", col("customer_street").cast("string")) \
                    .withColumn("customer_city", col("customer_city").cast("string")) \
                    .withColumn("customer_state", col("customer_state").cast("string")) \
                    .withColumn("customer_zipcode", col("customer_zipcode").cast("integer")) \
                    .withColumn("record_hash", col("record_hash"))
                        


In [0]:
%sql
describe retails.silver.customers_cleaned;

In [0]:
# make ready to upsert silver table customers_cleaned
silver_df = cdc_df \
            .withColumn("customer_id", col("customer_id").cast("bigint")) \
            .withColumn("customer_zipcode", col("customer_zipcode").cast("bigint")) \
            .withColumn("batch_id", col("batch_id").cast("string"))


In [0]:
# upsert customers_cleaned table
MERGE_UPDATE = """
    MERGE INTO retails.silver.customers_cleaned t
    USING global_temp.customers_cleaned_vw s
    ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.record_hash != t.record_hash AND s.op='UPDATE' THEN
        UPDATE SET t.customer_fname = s.customer_fname, t.customer_lname = s.customer_lname, t.customer_email = s.customer_email, t.customer_password = s.customer_password, t.customer_street = s.customer_street, t.customer_city =s.customer_city, t.customer_state = s.customer_state, t.customer_zipcode = s.customer_zipcode, t.is_deleted = false, t.updated_ts = current_timestamp(), t.op= s.op, t.record_hash = s.record_hash
    
    WHEN MATCHED AND s.is_deleted AND s.op='DELETE' THEN
        UPDATE SET t.is_deleted = true, t.updated_ts = current_timestamp(), t.op= s.op
    
    WHEN NOT MATCHED AND s.op='INSERT' THEN
        INSERT (
            customer_id,
            customer_fname,
            customer_lname,
            customer_email,
            customer_password,
            customer_street,
            customer_city,
            customer_state,
            customer_zipcode,
            is_deleted,
            ingestion_ts,
            ingestion_dt,
            source_system,
            source_file_name,
            batch_id,
            run_id,
            op,
            record_hash,
            created_ts,
            updated_ts
                    )
        VALUES(
            s.customer_id,
            s.customer_fname,
            s.customer_lname,
            s.customer_email,
            s.customer_password,
            s.customer_street,
            s.customer_city,
            s.customer_state,
            s.customer_zipcode,
            s.is_deleted,
            s.ingestion_ts,
            s.ingestion_dt,
            s.source_system,
            s.source_file_name,
            s.batch_id,
            s.run_id,
            s.op,
            s.record_hash,
            current_timestamp(),
            current_timestamp()
            )
    """

# spark.sql(merge_query).show()



In [0]:
def upsert_to_silver(batch_df, batch_id):

    batch_df = batch_df.cache()
    batch_df.createOrReplaceGlobalTempView(
        "customers_cleaned_vw"
    )
    spark.sql(MERGE_UPDATE)

    batch_df.unpersist()

In [0]:
silver_df = silver_df.drop("event_ts")
# silver_df.printSchema()

In [0]:

silver_df.writeStream \
    .format("delta") \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", "dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/customers_cleaned/") \
    .trigger(once=True) \
    .start() \
    .awaitTermination()

In [0]:
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/customers_cdc/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/customers_cleaned/", True)

# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/customers_cleaned")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/customers_cdc")